# Stage 13 - the RL reranker at a matched training budget

Trains the policy-gradient and cross-entropy objectives for four epochs each, then
continues both to eight, and scores them on the Stage 6 bench.
Details: `docs/stage13_rl_budget.md`.

## Setup

In [ ]:
# project folder on Drive
PROJECT_DIR = '/content/drive/MyDrive/RAG chunk optimize'

from google.colab import drive
drive.mount('/content/drive')

import os, sys
if not os.path.isfile(os.path.join(PROJECT_DIR, 'config.py')):
    raise RuntimeError(f'no config.py under {PROJECT_DIR!r} - fix PROJECT_DIR above.')

os.environ['RAG_DATA_ROOT'] = PROJECT_DIR + '/artifacts'
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)

import config as C
C.ensure_dirs()
print('project:', PROJECT_DIR)
print(C.summary())

## Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## Check the runtime

Stops here if no GPU is available.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('no GPU - switch to a GPU runtime (Runtime > Change runtime type).')
print('GPU:', torch.cuda.get_device_name(0))

## Training smoke check

Two steps per arm, written to a temporary folder and deleted.

In [ ]:
!python scripts/30_train_rl_budget.py --smoke

## Train the four arms

`ce4` and `rl4` start from the Stage 8 weights; `ce8` and `rl8` continue from them.
Each arm writes a completion marker, so a rerun skips finished arms.

In [ ]:
!python scripts/30_train_rl_budget.py

## Dev bench

Provenance only. The claim bench runs whatever this says.

In [ ]:
!python scripts/31_eval_rl_budget.py --dev

## Stage 6 bench

Fixed 15/0, six rerankers over the same pool. Checkpointed per config, so a rerun
resumes.

In [ ]:
!python scripts/31_eval_rl_budget.py

## Results

In [ ]:
import json, os, pathlib
from IPython.display import Markdown, display

latest = pathlib.Path(os.environ['RAG_DATA_ROOT']) / 'results' / 'latest'
summary = latest / 'stage13_summary.md'
if summary.exists():
    display(Markdown(summary.read_text(encoding='utf-8')))
    print(json.dumps(json.loads((latest / 'stage13_verdict.json').read_text(encoding='utf-8')),
                     indent=2))
else:
    print('no Stage 13 summary - the evaluation did not finish.')

## Verdicts

`INVALID`, `FAILED-OPTIMISATION`, `INCONCLUSIVE-BUDGET`, `RL-BETTER`, `CE-BETTER`
and `TIE` are defined in `docs/stage13_rl_budget.md`. Archiving is manual:
`results/latest/` also holds other stages' files.